# Part 3: Hands-On FALCON — Watch Active Learning Happen

## What We Know (from Part 2)
- FALCON wraps a surrogate model (GPR) around an exact calculator
- It decides when to call DFT based on uncertainty ε
- It clusters training data and retrains only the nearest model
- The OTF loop runs during MD

## What We'll Do Now
1. **Run** a FALCON simulation on a Pt₅₅ cluster
2. **Observe** the training data grow in real-time
3. **Change** ε and see how it affects DFT calls
4. **Change** model_size and see how it affects training time
5. **Diagnose** the health of the simulation
6. **Predict** outcomes before running, then verify

## Baseline: The Default FALCON Simulation

We'll run FALCON on a Pt₅₅ cluster at 600 K with:
- ε = 0.10 eV (default threshold)
- model_size = 200 (default clustering)

**Before we run it, predict:**
1. How many DFT calls in the first 1000 MD steps?
2. Will the training data growth plateau?

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import os
import time
import numpy as np
import matplotlib.pyplot as plt

from ase import units
from ase.build import bulk
from ase.calculators.emt import EMT
from ase.calculators.singlepoint import SinglePointCalculator
from ase.optimize import QuasiNewton
from ase.md.langevin import Langevin

from falcon_md.otf_calculator import FALCON
from falcon_md.models.agox_models import GPR as FalconGPR

# ============================================================
# WINDOWS PATCHES  (re-run after every kernel restart)
# ============================================================
if not hasattr(os, "sched_getaffinity"):
    os.sched_getaffinity = lambda pid: set(range(os.cpu_count() or 1))
os.environ["RAY_raylet_start_wait_time_s"] = "120"

# ============================================================
# SETUP
# ============================================================
T = 600
accuracy_e = 0.10
n_steps = 100
model_size = 200

os.makedirs("outputs", exist_ok=True)

# ============================================================
# INITIAL STRUCTURE
# ============================================================
atoms = bulk('Pt', 'fcc', a=3.92, cubic=True).repeat((3, 3, 3))
atoms.rattle(0.1)          # break symmetry so QN has work to do
exact_calc = EMT()
atoms.calc = exact_calc

# ============================================================
# GEOMETRY OPTIMIZATION FOR INITIAL TRAINING DATA
# ============================================================
training_frames = []

def collect(atoms):
    # Snapshot the current state and freeze its energy/forces into a
    # SinglePointCalculator, so AGOX's EnergyFilter can call
    # get_potential_energy() on each frame later.
    frame = atoms.copy()
    frame.calc = SinglePointCalculator(
        frame,
        energy=atoms.get_potential_energy(),
        forces=atoms.get_forces(),
    )
    training_frames.append(frame)

qn = QuasiNewton(atoms, logfile="outputs/opt_baseline.log")
qn.attach(collect, interval=1, atoms=atoms)
qn.run(fmax=0.05, steps=20)

training_data = training_frames
print(f"Initial training data: {len(training_data)} structures")

# ============================================================
# FALCON CALCULATOR
# ============================================================
atoms.calc = FALCON(
    model=FalconGPR(atoms),   # FALCON's GPR wrapper (use_ray=True)
    calc=exact_calc,          # signature uses `calc`, not `exact_calc`
    training_data=training_data,
    accuracy_e=accuracy_e,    # signature uses `accuracy_e`, not `accuracy`
    modelsize=model_size,     # signature uses `modelsize`, not `model_size`
    train_log=False,
)

# ============================================================
# RUN MD & TRACK
# ============================================================
energies = []
n_training_at_step = []
n_atoms = len(atoms)

dyn = Langevin(
    atoms,
    1 * units.fs,
    temperature_K=T,
    friction=0.002,
    trajectory="outputs/md_baseline.traj",
    logfile="outputs/md_baseline.log",
)

print(f"Running FALCON OTF-MD: T={T} K, ε={accuracy_e} eV, "
      f"modelsize={model_size}")
print(f"System: {n_atoms} atoms, {n_steps} MD steps")
print()

t_start = time.time()

for step in range(n_steps):
    dyn.run(1)
    energies.append(atoms.get_potential_energy())
    n_training_at_step.append(len(atoms.calc.training_data))

t_total = time.time() - t_start

print("Simulation complete!")
print(f"  Total wall time: {t_total:.1f} s")
print(f"  Final training structures: {len(atoms.calc.training_data)}")
print(f"  Final energy: {energies[-1]:.4f} eV")

Initial training data: 9 structures
Using C:\Users\carli/tmp/ray as Ray temporary folder because the given or generated path is too long.


2026-09-22 23:00:11,021	INFO worker.py:2024 -- Started a local Ray instance.


Ray Resources
	CPU = 12.0
	node:__internal_head__ = 1.0
	node:127.0.0.1 = 1.0
	object_store_memory = 6000000000.0
	memory = 24000000000.0
Attempt 0: Pool started successfully.


(raylet) Stack (most recent call first):
(raylet)   File "C:\Users\carli\AppData\Local\miniconda\envs\ci\Lib\site-packages\ray\_private\worker.py", line 627 in job_logging_config
(raylet)   File "C:\Users\carli\AppData\Local\miniconda\envs\ci\Lib\site-packages\ray\_private\worker.py", line 2807 in disconnect
(raylet)   File "C:\Users\carli\AppData\Local\miniconda\envs\ci\Lib\site-packages\ray\_private\worker.py", line 2118 in shutdown
(raylet)   File "C:\Users\carli\AppData\Local\miniconda\envs\ci\Lib\site-packages\ray\_private\worker.py", line 1170 in wrapper
(raylet)   File "C:\Users\carli\AppData\Local\miniconda\envs\ci\Lib\site-packages\ray\_private\client_mode_hook.py", line 134 in wrapper
(raylet) access violation
(raylet) 
(raylet)   File "C:\Users\carli\AppData\Local\miniconda\envs\ci\Lib\site-packages\ray\_private\worker.py"Windows fatal exception: access violationStack (most recent call first):
(raylet)   File 
(raylet) 
(raylet)   File "C:\Users\carli\AppData\Local\miniconda


You are using the Gaussian Process Regression model implemented in AGOX. If you use this model in a publication, please cite:
Mads-Peter V. Christiansen, Nikolaj Rønne, Bjørk Hammer, "Atomistic Global Optimization X: A Python package for optimization of atomistic structures", J. Chem. Phys. 157, 054701 (2022),
<https://arxiv.org/abs/2204.01451>

If you are using FALCON in a publication, please cite: 
 Felis, N., Dononelli, W. (2025). FALCON: fast active learning for machine learning potentials in atomistic and ab initio molecular dynamics simulations. npj Comput. Mater., https://doi.org/10.1038/s41524-025-01897-8
Running FALCON OTF-MD: T=600 K, ε=0.1 eV, modelsize=200
System: 108 atoms, 100 MD steps


============================ Step: 1 ============================

Total structures in training data: 9
1 active Models.
(9, 1, 60)
[0 0 0 0 0 0 0 0 0]
Cluster 0 contains 9 objects.
Clustering of 9 took 2.3270814418792725 s.
<class 'agox.models.GPR.GPR.GPR'>


...

------ Training Info ------

Training data size: 19

Neg. log marginal likelihood.: -6.78

Hyperparameter optimizations: 12

Step 10: Model 0 trained with 19 Structures (4.052531719207764 s). (1 Training steps.) 


============================ Step: 11 ============================

Total structures in training data: 19
Model 0...
Predict energy (0.16164827346801758 s)
Predict energy_uncertainty (0.17601823806762695 s)
Predict forces (0.5765097141265869 s)
Predict forces_uncertainty (0.7066948413848877 s)
E = -0.013764497256929742 (Model 0)
Estd = 0.027143082655994993 (Model 0)
F = 0.03654708508493055 (Model 0)
Fstd = 0.0037921172037623328 (Model 0)

Model 0 has the lowest uncertainty.
E = -0.013764497256929742
Estd = 0.027143082655994993
Fmax = 0.03654708508493055
Fstd = 0.0037921172037623328

Step 11: Model not trained.


============================ Step: 12 ============================

Total structures in training data: 19
Model 0...
Predict energy (0.16024112701416016 s)
Predict energy_uncertainty (0.12778329849243164 s)
Predict forces (0.7155823707580566 s)
Predict forces_uncertainty (1.301651000976562

## Visualizing What Just Happened

Let's plot the energy trajectory and the training data growth together.

In [ ]:
# ============================================================
# PLOT BASELINE RESULTS
# ============================================================
n_steps_ran = len(n_training_at_step)
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Energy
axes[0].plot(energies, 'r-', alpha=0.6, linewidth=0.5)
axes[0].set_ylabel('Energy (eV)', fontsize=12)
axes[0].set_title(f'FALCON OTF-MD: Pt, T={T} K, ε={accuracy_e} eV, '
                  f'modelsize={model_size}',
                  fontsize=14)
axes[0].grid(True, alpha=0.3)

# Training data growth
axes[1].plot(n_training_at_step, 'b-', linewidth=2)
axes[1].set_xlabel('MD step', fontsize=12)
axes[1].set_ylabel('Training structures', fontsize=12)
axes[1].set_title('Growth of training data', fontsize=14)
axes[1].grid(True, alpha=0.3)

# Mark where the training set stops growing.
# Look at the last 20% of the run and find the first step from which
# the count never changes again.
tail_start = int(0.8 * n_steps_ran)
final_count = n_training_at_step[-1]
plateau_start = None
for i in range(tail_start, n_steps_ran):
    if n_training_at_step[i] == final_count:
        # Walk backwards to find the earliest index with this value.
        j = i
        while j > 0 and n_training_at_step[j - 1] == final_count:
            j -= 1
        plateau_start = j
        break

if plateau_start is not None and plateau_start < n_steps_ran - 1:
    axes[1].axvline(x=plateau_start, color='green', linestyle='--',
                    alpha=0.7, label=f'Plateau at step {plateau_start}')
    axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/falcon_baseline.png', dpi=150)
plt.show()

# ============================================================
# QUANTIFY THE TRAINING PHASES
# ============================================================
# Divide the run into thirds, regardless of how many steps we ran.
i1 = n_steps_ran // 3
i2 = 2 * n_steps_ran // 3

n_early = n_training_at_step[i1] - n_training_at_step[0]
n_mid   = n_training_at_step[i2] - n_training_at_step[i1]
n_late  = n_training_at_step[-1] - n_training_at_step[i2]

print(f"Training phases (run length: {n_steps_ran} MD steps):")
print(f"  Steps     0 - {i1:>5}: +{n_early} structures")
print(f"  Steps {i1:>5} - {i2:>5}: +{n_mid} structures")
print(f"  Steps {i2:>5} - {n_steps_ran:>5}: +{n_late} structures")

## How Does ε Affect the Simulation?

**Before running, predict:**
- If ε = 0.20 eV: more or fewer training structures?
- If ε = 0.05 eV: more or fewer training structures?
- Is the relationship linear?

Let's test our predictions.

In [ ]:
# ============================================================
# IMPORTS (just in case ;) )
# ============================================================
import os
import time
import numpy as np
import matplotlib.pyplot as plt

from ase import units
from ase.build import bulk
from ase.calculators.emt import EMT
from ase.calculators.singlepoint import SinglePointCalculator
from ase.io import read
from ase.optimize import QuasiNewton
from ase.md.langevin import Langevin

from falcon_md.otf_calculator import FALCON
from falcon_md.models.agox_models import GPR as FalconGPR

# Windows patches
if not hasattr(os, "sched_getaffinity"):
    os.sched_getaffinity = lambda pid: set(range(os.cpu_count() or 1))
os.environ["RAY_raylet_start_wait_time_s"] = "120"

os.makedirs("outputs", exist_ok=True)

# ============================================================
# SWEEP OVER ACCURACY THRESHOLDS
# ============================================================
T = 600
n_steps_short = 100          # shorter for the sweep
thresholds = [0.05, 0.10, 0.15, 0.20]
model_size = 200

results_eps = {}

def generate_training_data(tag):
    """Build a fresh Pt supercell and return a small training set."""
    atoms = bulk('Pt', 'fcc', a=3.92, cubic=True).repeat((3, 3, 3))
    atoms.rattle(0.1)
    atoms.calc = EMT()

    frames = []
    def collect(atoms):
        frame = atoms.copy()
        frame.calc = SinglePointCalculator(
            frame,
            energy=atoms.get_potential_energy(),
            forces=atoms.get_forces(),
        )
        frames.append(frame)

    qn = QuasiNewton(atoms, logfile=f"outputs/opt_{tag}.log")
    qn.attach(collect, interval=1, atoms=atoms)
    qn.run(fmax=0.05, steps=20)

    return atoms, frames

for eps in thresholds:
    print(f"\n{'='*50}")
    print(f"Running ε = {eps} eV")
    print(f"{'='*50}")

    atoms, training_data = generate_training_data(f"eps{eps}")
    print(f"  Initial training data: {len(training_data)} structures")

    exact_calc = EMT()
    atoms.calc = FALCON(
        model=FalconGPR(atoms),
        calc=exact_calc,
        training_data=training_data,
        accuracy_e=eps,
        modelsize=model_size,
        train_log=False,        # suppress per-step printout
    )

    t_start = time.time()
    dyn = Langevin(atoms, 1 * units.fs, temperature_K=T, friction=0.002)
    for step in range(n_steps_short):
        dyn.run(1)
    t_total = time.time() - t_start

    n_train = len(atoms.calc.training_data)
    results_eps[eps] = {
        'n_train': n_train,
        'time': t_total,
        'n_steps': n_steps_short,
    }

    print(f"  Training structures: {n_train}")
    print(f"  Total time: {t_total:.1f} s")

# ============================================================
# PLOT RESULTS
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

eps_vals = list(results_eps.keys())
n_trains = [results_eps[e]['n_train'] for e in eps_vals]
times    = [results_eps[e]['time']    for e in eps_vals]

axes[0].plot(eps_vals, n_trains, 'o-', color='steelblue',
             linewidth=2, markersize=10)
axes[0].set_xlabel('Accuracy threshold ε (eV)', fontsize=12)
axes[0].set_ylabel('Training structures', fontsize=12)
axes[0].set_title('How ε affects DFT calls', fontsize=14)
axes[0].grid(True, alpha=0.3)
axes[0].invert_xaxis()   # lower ε on the right

axes[1].plot(eps_vals, times, 's-', color='coral',
             linewidth=2, markersize=10)
axes[1].set_xlabel('Accuracy threshold ε (eV)', fontsize=12)
axes[1].set_ylabel('Total wall time (s)', fontsize=12)
axes[1].set_title('How ε affects simulation time', fontsize=14)
axes[1].grid(True, alpha=0.3)
axes[1].invert_xaxis()

plt.tight_layout()
plt.savefig('outputs/eps_sweep.png', dpi=150)
plt.show()

print("\nSummary:")
for eps in eps_vals:
    print(f"  ε = {eps:.2f} eV → {results_eps[eps]['n_train']} structures, "
          f"{results_eps[eps]['time']:.1f} s")

## How Does Clustering Affect Performance?

**Before running, predict:**
- model_size = 100: faster or slower than no clustering?
- model_size = 500: faster or slower than no clustering?
- Which model_size gives the fastest simulation?

Let's test.

In [ ]:
# ============================================================
# SWEEP OVER MODEL SIZES
# ============================================================
model_sizes = [100, 200, 500, None]  # None = no clustering
results_ms = {}

for ms in model_sizes:
    label = f"size={ms}" if ms else "no clustering"
    print(f"\n{'='*50}")
    print(f"Running: {label}")
    print(f"{'='*50}")
    
    # Fresh structure
    atoms = bulk('Pt', 'fcc', a=3.92, cubic=True).repeat((3, 3, 3))
    exact_calc = EMT()
    atoms.calc = exact_calc
    
    # Optimize
    qn = QuasiNewton(atoms, trajectory=f'outputs/opt_ms{ms}.traj')
    qn.run(fmax=0.05)
    training_data = read(f'outputs/opt_ms{ms}.traj@0:1')
    
    # FALCON
    atoms.calc = FALCON(
        model=GPR(atoms),
        exact_calc=exact_calc,
        training_data=training_data,
        accuracy_e=0.10,
        model_size=ms
    )
    
    # Run
    n_steps_short = 3000
    t_start = time.time()
    
    dyn = Langevin(atoms, 1*units.fs, temperature_K=T, friction=0.002)
    for step in range(n_steps_short):
        dyn.run(1)
    
    t_total = time.time() - t_start
    n_train = len(atoms.calc.training_data)
    
    results_ms[label] = {
        'n_train': n_train,
        'time': t_total,
    }
    
    print(f"  Training structures: {n_train}")
    print(f"  Total time: {t_total:.1f} s")

# ============================================================
# PLOT RESULTS
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

labels = list(results_ms.keys())
n_trains = [results_ms[l]['n_train'] for l in labels]
times = [results_ms[l]['time'] for l in labels]

axes[0].bar(labels, n_trains, color='steelblue')
axes[0].set_ylabel('Training structures', fontsize=12)
axes[0].set_title('Training data volume', fontsize=14)
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(labels, times, color='coral')
axes[1].set_ylabel('Total wall time (s)', fontsize=12)
axes[1].set_title('Simulation time', fontsize=14)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('outputs/model_size_sweep.png', dpi=150)
plt.show()

print("\nSummary:")
for label in labels:
    print(f"  {label:15s} → {results_ms[label]['n_train']} structures, "
          f"{results_ms[label]['time']:.1f} s")

## Watching FALCON Learn in Real Time

Let's run a shorter simulation with a live-updating plot.

This shows the active learning process as it happens.

In [ ]:
# ============================================================
# REAL-TIME PLOT (works in Jupyter)
# ============================================================
from IPython.display import display, clear_output

# Setup
atoms = bulk('Pt', 'fcc', a=3.92, cubic=True).repeat((3, 3, 3))
exact_calc = EMT()
atoms.calc = exact_calc

qn = QuasiNewton(atoms, trajectory='outputs/opt_live.traj')
qn.run(fmax=0.05)
training_data = read('outputs/opt_live.traj@0:1')

atoms.calc = FALCON(
    model=GPR(atoms),
    exact_calc=exact_calc,
    training_data=training_data,
    accuracy_e=0.10,
    model_size=200
)

dyn = Langevin(atoms, 1*units.fs, temperature_K=T, friction=0.002)

# Live plot
fig, axes = plt.subplots(2, 1, figsize=(10, 6))
energies_live = []
n_train_live = []

for step in range(2000):
    dyn.run(1)
    energies_live.append(atoms.get_potential_energy())
    n_train_live.append(len(atoms.calc.training_data))
    
    if step % 100 == 0:
        clear_output(wait=True)
        
        axes[0].clear()
        axes[0].plot(energies_live, 'r-', alpha=0.6)
        axes[0].set_ylabel('Energy (eV)')
        axes[0].set_title(f'Step {step}: E = {energies_live[-1]:.3f} eV')
        
        axes[1].clear()
        axes[1].plot(n_train_live, 'b-', linewidth=2)
        axes[1].set_ylabel('Training structures')
        axes[1].set_xlabel('MD step')
        axes[1].set_title(f'Training data: {n_train_live[-1]} structures')
        
        plt.tight_layout()
        display(fig)
        plt.close()

print(f"\nFinal: {n_train_live[-1]} training structures")

## Is My Simulation Healthy?

Not every FALCON run goes smoothly. Here's how to diagnose problems.

In [ ]:
# ============================================================
# HEALTH CHECK
# ============================================================
def diagnose_falcon(atoms, energies, n_training):
    E_std = np.std(energies)
    E_drift = abs(np.mean(energies[-100:]) - np.mean(energies[:100]))

    print("\n1. Energy stability:")
    print(f"   Energy std:   {E_std:.4f} eV")
    print(f"   Energy drift: {E_drift:.4f} eV")
    print("   WARNING: significant energy drift!" if E_drift > 0.1 else "   Energy is stable")

    n_final = n_training[-1]
    n_growth = n_training[-1] - n_training[len(n_training) // 2]
    print("\n2. Training data growth:")
    print(f"   Final training structures: {n_final}")
    print(f"   Growth in second half:     {n_growth}")
    if n_growth > n_final * 0.5:
        print("   WARNING: training data still growing rapidly — model may not have converged")
        print("   Consider increasing ε or running longer")
    else:
        print("   Training data has stabilized")

    print("\n3. System health:")
    pos = atoms.get_positions()
    max_disp = np.max(np.linalg.norm(pos - pos.mean(axis=0), axis=1))
    print(f"   Number of atoms: {len(atoms)}")
    print(f"   Max atomic displacement from center: {max_disp:.2f} Å")
    print("   WARNING: atoms may have evaporated!" if max_disp > 15 else "   System is intact")

# Run diagnosis
diagnose_falcon(atoms, energies, n_training_at_step)

## From Pt₅₅ to Li₃OCl/Br

The Pt₅₅ demo is simple: one element, no electrostatics, no defects.

Your system is harder: five elements, ionic, defect-rich.

### What Changes

| Feature | Pt₅₅ Demo | Li₃OCl/Br |
|:---|:---|:---|
| **Elements** | 1 (Pt) | 5 (Li, O, Cl, Br, Al) |
| **Electrostatics** | None | Strong (ionic conductor) |
| **Defects** | None | Vacancies, interstitials, cascades |
| **Training data** | 1 optimized structure | 50-100 DFT frames |
| **Exact calculator** | EMT (milliseconds) | VASP/QE (minutes to hours) |
| **Accuracy threshold** | 0.10 eV | 0.05-0.10 eV |
| **Model size** | 200 | 200-500 |

### The Adaptation Workflow

1. Prepare initial training data
training_data = read('li3oclbr_pristine.traj')  # 50-100 DFT frames

2. FALCON setup with DFT as exact calculator
```
from ase.calculators.vasp import Vasp
exact_calc = Vasp(...)  # Your DFT settings

atoms.calc = FALCON(
    model=GPR(atoms),
    exact_calc=exact_calc,
    training_data=training_data,
    accuracy_e=0.10,
    model_size=300  # Larger for multi-element systems
)
```

3. Run short OTF-MD at 600 K
4. Post-process to find uncertain structures
5. Add them to DFT training set
6. Train MACE for production

## Summary: What We Just Did

| Experiment | What We Learned |
|:---|:---|
| **Baseline run** | Training data grows rapidly, then plateaus |
| **ε sweep** | Lower ε → more DFT calls, longer simulation |
| **model_size sweep** | Clustering beats no clustering; 200 is optimal for Pt₅₅ |
| **Real-time plot** | Active learning is fast then confident |
| **Health check** | Energy drift and training growth diagnose problems |
| **Li₃OCl/Br adaptation** | More elements, more DFT, more care |

## The Complete Workshop So Far
Part 0: Setup

Part 1a: Built a neuron

Part 1b: Computed forces via autograd

Part 1c: Trained a NN potential

Part 2: Understood FALCON architecture

Part 3: Ran FALCON and analyzed results ← WE ARE HERE

## Next: Part 4 — Clustering Deep Dive

In Part 4, we'll:
1. Extract the training data from our FALCON run
2. Cluster it with SOAP descriptors
3. Visualize the PES regions
4. Understand which configurations matter most
